# AI-Generated Summaries — Task 1 & Task 2 (Generation)
**NLP Final Project — Dialectal Robustness**

Generates the LLM summaries used by the rest of the project:

- **Task 1 – Generation Bias (unconstrained)**: summarize each article with no dialect
  instruction, so we can measure how often each model *drifts* to American English
  regardless of the source dialect.
- **Task 2 – Evaluation Bias (constrained)**: summarize each article three times per model,
  explicitly targeting US / UK / AUS English, for downstream BERTScore / LLM-judge / LSR scoring.

Reads articles from `summary_articles/` and writes checkpointed JSONL + CSV results to `summaries/`.

## Model choice

The proposal calls for one closed-source and one open-source model:

| Role | Proposal | Used here | Why |
|---|---|---|---|
| Closed-source | Gemini | **`gemini-2.5-flash`** via the Google AI Studio API | Free tier (generous daily quota) — no cost for a project of this size. |
| Open-source | Llama 4 Maverick | **`openai/gpt-oss-120b`** via **Groq** | Llama 4 Maverick is not available (free or otherwise) on Groq's current catalog — confirmed by querying `groq_client.models.list()` directly, which returned only `llama-3.1-8b-instant`, `llama-3.3-70b-versatile`, and two small prompt-guard models under the Llama name. Of everything actually available on a free Groq key, **`openai/gpt-oss-120b`** (OpenAI's Apache-2.0-licensed open-weight model, 120B MoE) is the strongest — stronger than `llama-3.3-70b-versatile`, the closest same-family fallback. It's genuinely open-weight and free via Groq, just not from Meta. |

Get a free Groq key at **console.groq.com/keys** and a free Gemini key at **aistudio.google.com/app/apikey**.
If `gpt-oss-120b`'s free-tier rate limits are ever too tight, `llama-3.3-70b-versatile`
(Meta, dense, also free on Groq) is a drop-in fallback — just change `OPEN_MODEL` below.
Re-run the model-list cell any time to see what Groq currently has available on your key —
their free-tier catalog has already changed once during this project.

In [15]:
# %pip install -U google-genai groq pandas tqdm
# Run the line above (uncommented) once if any of these aren't installed yet.

In [ ]:
import json
import os
import re
import time
import textwrap
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from tqdm.notebook import tqdm

from google import genai
from google.genai import types as genai_types
from groq import Groq

from helpers.dialect_classifier import OUTLET_TO_DIALECT
from helpers.spelling_markers import load_spelling_markers, compute_lsr, strip_quotes

## Configuration
Fill in your API keys below before running any other cells.

In [ ]:
# ── Test mode ────────────────────────────────────────────────────────────
# True  -> only process the first TEST_LIMIT article(s), but still call every
#          model (and, for Task 2, every target dialect) on them — this is a
#          smoke test of BOTH APIs, not just whichever model happens to be first.
# False -> generate summaries for every article / model / dialect combination
TEST_RUN = False
TEST_LIMIT = 1  # number of articles to use when TEST_RUN is True

# ── API Keys ─────────────────────────────────────────────────────────────
GEMINI_API_KEY = "AQ.Ab8RN6L--t1NYVPhP0eX1MbI685_TFBkbyFIW2mAzcrkoO_ZOQ"   # aistudio.google.com/app/apikey
GROQ_API_KEY   = "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"     # console.groq.com/keys

# ── Models ───────────────────────────────────────────────────────────────
GEMINI_MODEL = "gemini-2.5-flash"     # closed-source
OPEN_MODEL   = "openai/gpt-oss-120b"  # open-source, free via Groq (Llama 4 Maverick isn't on Groq's catalog anymore)

# ── Paths ────────────────────────────────────────────────────────────────
ARTICLES_DIR  = "./summary_articles"
SUMMARIES_DIR = "./summaries"
Path(SUMMARIES_DIR).mkdir(exist_ok=True)

# ── Run settings ─────────────────────────────────────────────────────────
REQUEST_DELAY = 1.5   # seconds between API calls (stay well under free-tier rate limits)

# Per-run call cap per model — a proactive safety margin so a run stops just BEFORE the
# hard quota wall instead of burning retries against it. Gemini's free tier on this key is
# capped at 20 requests/DAY (confirmed via a 429 RESOURCE_EXHAUSTED response), so 18 leaves
# headroom for the model-list-check cell. None = uncapped (rely on reactive detection only).
# This is PER RUN, not per day — if you run the notebook more than once in a day, the real
# daily wall still applies and QuotaExceededError handling below will catch it regardless.
MAX_CALLS_PER_MODEL_PER_RUN = {"gemini": 18, "gpt_oss": None}
MAX_TOKENS    = 512
TEMPERATURE   = 0.3

print("Config loaded.")
print(f"  TEST_RUN      : {TEST_RUN} (first {TEST_LIMIT} article(s), all models/dialects)" if TEST_RUN else "  TEST_RUN      : False (full run)")
print(f"  Articles dir  : {ARTICLES_DIR}")
print(f"  Summaries dir : {SUMMARIES_DIR}")
print(f"  Gemini model  : {GEMINI_MODEL}")
print(f"  Open model    : {OPEN_MODEL}")
print(f"  Call caps     : {MAX_CALLS_PER_MODEL_PER_RUN}")

Config loaded.
  TEST_RUN      : False (full run)
  Articles dir  : ./summary_articles
  Summaries dir : ./summaries
  Gemini model  : gemini-2.5-flash
  Open model    : openai/gpt-oss-120b


## Initialise API Clients

In [18]:
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
groq_client   = Groq(api_key=GROQ_API_KEY)

print(f"Gemini : {GEMINI_MODEL}  \u2713")
print(f"Open   : {OPEN_MODEL} via Groq  \u2713")

Gemini : gemini-2.5-flash  ✓
Open   : openai/gpt-oss-120b via Groq  ✓


In [19]:
# Optional sanity check — confirms both model IDs are actually reachable with your keys.
# Also useful on its own: Groq's free-tier catalog changes over time (Llama 4 Maverick/Scout
# were removed from it at some point after this notebook was first written), so it's worth
# re-running this before a full batch run to confirm OPEN_MODEL is still listed.
print("=== Gemini models supporting generateContent ===")
for m in gemini_client.models.list():
    supported = getattr(m, "supported_actions", None) or getattr(m, "supported_generation_methods", [])
    if "generateContent" in supported and "flash" in (m.name or ""):
        print(f"  {m.name}")

print("\n=== Groq chat models (excludes audio/TTS/guard/compound-agent models) ===")
EXCLUDE = ("whisper", "orpheus", "prompt-guard", "compound", "allam")
for m in groq_client.models.list().data:
    if not any(k in m.id for k in EXCLUDE):
        print(f"  {m.id}")

=== Gemini models supporting generateContent ===
  models/gemini-2.5-flash
  models/gemini-2.0-flash
  models/gemini-2.0-flash-001
  models/gemini-2.0-flash-lite-001
  models/gemini-2.0-flash-lite
  models/gemini-2.5-flash-preview-tts
  models/gemini-flash-latest
  models/gemini-flash-lite-latest
  models/gemini-2.5-flash-lite
  models/gemini-2.5-flash-image
  models/gemini-3-flash-preview
  models/gemini-3.1-flash-lite-preview
  models/gemini-3.1-flash-lite
  models/gemini-3.1-flash-image-preview
  models/gemini-3.1-flash-image
  models/gemini-3.1-flash-lite-image
  models/gemini-3.5-flash
  models/gemini-3.5-flash-lite
  models/gemini-omni-flash-preview
  models/gemini-3.6-flash
  models/gemini-3.1-flash-tts-preview

=== Groq chat models (excludes audio/TTS/guard/compound-agent models) ===
  openai/gpt-oss-120b
  llama-3.1-8b-instant
  openai/gpt-oss-20b
  llama-3.3-70b-versatile
  qwen/qwen3.6-27b
  openai/gpt-oss-safeguard-20b


## Load Articles
Reads the flat article JSON files directly inside `summary_articles/` (each file already
carries its own `topic`, `source`, `title`, and `full_text` fields — no folder-structure
parsing needed). The nested `summary_articles/<topic>/<outlet>/` subfolders are leftover
from an earlier fetch and are intentionally **not** read here — some of their files are
mislabeled (wrong topic/content pairing), while the flat files are the curated, current set.

In [ ]:
def load_articles(articles_dir: str = ARTICLES_DIR) -> list[dict]:
    """Load the flat, top-level article JSON files from articles_dir (non-recursive).

    Drops exact-duplicate articles (same normalized text, e.g. re-fetched under a
    different topic name) so we never spend API calls generating two "independent"
    sets of summaries for the same article. See summary_articles/_duplicates_archive/
    for what was already found and removed on disk — this is just a defensive second
    layer in case duplicates ever creep back in.
    """
    records = []
    seen_hashes = set()

    for fpath in sorted(Path(articles_dir).glob("*.json")):
        try:
            with open(fpath, encoding="utf-8") as f:
                raw = json.load(f)
        except (json.JSONDecodeError, OSError):
            continue

        full_text = (raw.get("full_text") or raw.get("content_snippet") or raw.get("description") or "").strip()
        if not full_text:
            continue

        content_hash = hash(" ".join(full_text.split()).lower())
        if content_hash in seen_hashes:
            continue
        seen_hashes.add(content_hash)

        outlet  = raw.get("source", "")
        dialect = OUTLET_TO_DIALECT.get(outlet)
        if dialect is None:
            dialect = {"AU": "AUS", "GB": "UK", "US": "US"}.get(raw.get("country", ""), "US")

        records.append({
            "full_text": full_text,
            "dialect":   dialect,
            "outlet":    outlet,
            "topic":     raw.get("topic", raw.get("topic_slug", "")),
            "title":     raw.get("title", ""),
            "file":      str(fpath),
        })

    return records


all_articles = load_articles()

counts = Counter(a["outlet"] for a in all_articles)
print(f"Articles loaded : {len(all_articles)}")
for outlet, n in sorted(counts.items()):
    print(f"  {outlet:<10} {n} article(s)  (dialect: {OUTLET_TO_DIALECT.get(outlet, '?')})")
if all_articles:
    print(f"\nSample: [{all_articles[0]['dialect']}] {all_articles[0]['title']}")

## Prompt Templates

### Task 1 — Unconstrained
No dialect instruction; the model writes however it naturally defaults.

### Task 2 — Dialect-Constrained
Explicitly requests a specific dialect. We test all three (US / UK / AUS) for *every* article
regardless of the article's original dialect.

In [21]:
UNCONSTRAINED_PROMPT = (
    "Summarize the following news article in 3-5 sentences. "
    "Be factual and concise. Do not add information not in the article.\n\n"
    "Article:\n{text}\n\nSummary:"
)

DIALECT_INSTRUCTIONS = {
    "US":  "American English",
    "UK":  "British English",
    "AUS": "Australian English",
}

# Spelling cues per dialect — grounds the model in orthography, not stereotyped vocabulary
DIALECT_SPELLING_CUES = {
    "US":  "color, organize, realize, analyze, center, license, defense",
    "UK":  "colour, organise, realise, analyse, centre, licence, defence",
    "AUS": "colour, organise, realise, analyse, centre, licence, defence",
}

CONSTRAINED_PROMPT = (
    "Summarize the following news article in 3-5 sentences.\n\n"
    "Write as a professional journalist who naturally uses {dialect_label} spelling conventions "
    "(e.g. {spelling_cues}). Apply those spelling norms consistently throughout. "
    "Do NOT use stereotypical, archaic, or unusual regional vocabulary \u2014 "
    "the dialect difference should be subtle and reflect how a real {dialect_label} journalist would write, "
    "not a caricature.\n\n"
    "Article:\n{text}\n\nSummary:"
)

print("Prompt templates ready.")

Prompt templates ready.


## API Call Functions
`call_with_retry` wraps any API call with exponential back-off.
`summarize_gemini` and `summarize_open` return the summary string or raise on final failure.

In [ ]:
class QuotaExceededError(Exception):
    """A provider's free-tier quota is exhausted for now — retrying within seconds
    won't help (e.g. Gemini's free tier here is capped at 20 requests/DAY), so this
    is deliberately NOT retried by call_with_retry."""
    pass


def _is_quota_error(exc: Exception) -> bool:
    status = getattr(exc, "status_code", None) or getattr(exc, "code", None)
    if status == 429:
        return True
    msg = str(exc).lower()
    return any(s in msg for s in ("429", "resource_exhausted", "rate_limit_exceeded", "quota"))


def call_with_retry(fn, max_retries: int = 3, base_delay: float = 5.0):
    for attempt in range(max_retries):
        try:
            return fn()
        except QuotaExceededError:
            raise  # quota wall — burning retries against it wastes time for no benefit
        except Exception as exc:
            if attempt == max_retries - 1:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"    [retry {attempt+1}/{max_retries} in {delay:.0f}s] {exc}")
            time.sleep(delay)


def summarize_gemini(prompt: str) -> str:
    def _call():
        try:
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=genai_types.GenerateContentConfig(
                    max_output_tokens=MAX_TOKENS,
                    temperature=TEMPERATURE,
                    # gemini-2.5-flash "thinks" by default, and thinking tokens are drawn
                    # from the SAME max_output_tokens budget as the visible answer — without
                    # disabling it, responses can hit MAX_TOKENS after spending nearly the
                    # whole budget on invisible reasoning, truncating the summary mid-sentence.
                    # Confirmed directly: one live call spent 487/512 tokens on thinking and
                    # returned only "Two hikers, both in their early 20s, were successfully
                    # located in Kosciuszko National Park" (finish_reason=MAX_TOKENS). Setting
                    # thinking_budget=0 fixed it (finish_reason=STOP, complete summary).
                    thinking_config=genai_types.ThinkingConfig(thinking_budget=0),
                ),
            )
        except Exception as exc:
            if _is_quota_error(exc):
                raise QuotaExceededError(str(exc)) from exc
            raise
        return resp.text.strip()
    return call_with_retry(_call)


def summarize_open(prompt: str) -> str:
    def _call():
        try:
            resp = groq_client.chat.completions.create(
                model=OPEN_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                reasoning_effort="low",   # gpt-oss is also a reasoning model — same failure
                                          # mode as Gemini's thinking tokens is possible here
            )
        except Exception as exc:
            if _is_quota_error(exc):
                raise QuotaExceededError(str(exc)) from exc
            raise
        return resp.choices[0].message.content.strip()
    return call_with_retry(_call)


MODELS = {
    "gemini": summarize_gemini,
    "gpt_oss": summarize_open,   # free via Groq
}

print("API functions ready:", list(MODELS))

## Checkpoint Helpers
Summaries are appended line-by-line to a JSONL file. If a cell is interrupted and re-run,
already-completed (article, model[, dialect]) combos are skipped automatically.

In [ ]:
def append_record(record: dict, path: str) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: str) -> list[dict]:
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def done_keys(records: list[dict], key_fields: list[str]) -> set:
    return {tuple(r[k] for k in key_fields) for r in records}


# ── Quota-aware batching ──────────────────────────────────────────────────
# Shared across Task 1 AND Task 2 in this run, since they draw on the same daily
# quota. Once a model hits a quota wall (or its optional MAX_CALLS_PER_MODEL_PER_RUN
# cap), remaining calls to it are skipped instantly for the rest of THIS run — they
# stay absent from the JSONL (not marked done), so the next run picks them up right
# where this one left off. That's what makes this "batches that track previous work":
# no separate batch-index bookkeeping needed, the checkpoint file already IS it.
quota_exhausted: set[str] = set()
calls_made: dict[str, int] = defaultdict(int)
skipped_for_quota: dict[str, int] = defaultdict(int)


def model_available(model_name: str) -> bool:
    if model_name in quota_exhausted:
        return False
    cap = MAX_CALLS_PER_MODEL_PER_RUN.get(model_name)
    return cap is None or calls_made[model_name] < cap


print("Checkpoint helpers ready.")

## Task 1 — Unconstrained Summarization (Generation Bias)

For every article, both models produce a free-form summary. The LSR (Lexical Switch Rate)
computed below measures how often each model switches to American English regardless of
the source dialect.

In [ ]:
TASK1_PATH = os.path.join(SUMMARIES_DIR, "task1_unconstrained.jsonl")

existing_t1 = load_jsonl(TASK1_PATH)
done_t1     = done_keys(existing_t1, ["file", "model"])
print(f"Task 1 — already done: {len(existing_t1)} records ({len(done_t1)} unique (file, model) pairs)")

articles_t1 = all_articles[:TEST_LIMIT] if TEST_RUN else all_articles

todo = [
    (art, model_name)
    for art in articles_t1
    for model_name in MODELS
    if (art["file"], model_name) not in done_t1
]
print(f"Remaining        : {len(todo)}")

errors_t1 = []

for art, model_name in tqdm(todo, desc="Task 1"):
    if not model_available(model_name):
        skipped_for_quota[model_name] += 1
        continue

    prompt = UNCONSTRAINED_PROMPT.format(text=art["full_text"])
    try:
        summary = MODELS[model_name](prompt)
        calls_made[model_name] += 1
        record = {
            "task":            "unconstrained",
            "model":           model_name,
            "article_title":   art["title"],
            "topic":           art["topic"],
            "outlet":          art["outlet"],
            "article_dialect": art["dialect"],
            "file":            art["file"],
            "summary":         summary,
            "timestamp":       datetime.now(timezone.utc).isoformat(),
        }
        append_record(record, TASK1_PATH)
    except QuotaExceededError:
        quota_exhausted.add(model_name)
        skipped_for_quota[model_name] += 1
        print(f"  ⚠️  {model_name}: quota exhausted — skipping remaining {model_name} calls this run. Re-run later to pick up where this left off.")
    except Exception as exc:
        errors_t1.append({"file": art["file"], "model": model_name, "error": str(exc)})
        print(f"  ERROR [{model_name}] {art['title'][:50]}: {exc}")
    time.sleep(REQUEST_DELAY)

t1_records = load_jsonl(TASK1_PATH)
print(f"\nTask 1 complete — {len(t1_records)} summaries saved to {TASK1_PATH}")
if errors_t1:
    print(f"  Errors: {len(errors_t1)} (see `errors_t1` list)")
if any(skipped_for_quota.values()):
    print(f"  Skipped for quota: {dict(skipped_for_quota)} — not marked done, will resume next run")

## Task 2 — Dialect-Constrained Summarization (Evaluation Bias)

Each article is summarized **three times per model** — once constrained to US, UK, and AUS
English respectively. These summaries are scored downstream with BERTScore and LLM-as-a-Judge
(see `evaluate_on_summaries.py`).

In [ ]:
TASK2_PATH = os.path.join(SUMMARIES_DIR, "task2_constrained.jsonl")

existing_t2 = load_jsonl(TASK2_PATH)
done_t2     = done_keys(existing_t2, ["file", "model", "target_dialect"])
print(f"Task 2 — already done: {len(existing_t2)} records")

articles_t2 = all_articles[:TEST_LIMIT] if TEST_RUN else all_articles

todo2 = [
    (art, model_name, dialect)
    for art in articles_t2
    for model_name in MODELS
    for dialect in DIALECT_INSTRUCTIONS
    if (art["file"], model_name, dialect) not in done_t2
]
print(f"Remaining        : {len(todo2)}")

errors_t2 = []

for art, model_name, dialect in tqdm(todo2, desc="Task 2"):
    if not model_available(model_name):
        skipped_for_quota[model_name] += 1
        continue

    prompt = CONSTRAINED_PROMPT.format(
        text=art["full_text"],
        dialect_label=DIALECT_INSTRUCTIONS[dialect],
        spelling_cues=DIALECT_SPELLING_CUES[dialect],
    )
    try:
        summary = MODELS[model_name](prompt)
        calls_made[model_name] += 1
        record = {
            "task":            "constrained",
            "model":           model_name,
            "article_title":   art["title"],
            "topic":           art["topic"],
            "outlet":          art["outlet"],
            "article_dialect": art["dialect"],
            "target_dialect":  dialect,
            "file":            art["file"],
            "summary":         summary,
            "timestamp":       datetime.now(timezone.utc).isoformat(),
        }
        append_record(record, TASK2_PATH)
    except QuotaExceededError:
        quota_exhausted.add(model_name)
        skipped_for_quota[model_name] += 1
        print(f"  ⚠️  {model_name}: quota exhausted — skipping remaining {model_name} calls this run. Re-run later to pick up where this left off.")
    except Exception as exc:
        errors_t2.append({"file": art["file"], "model": model_name, "dialect": dialect, "error": str(exc)})
        print(f"  ERROR [{model_name}/{dialect}] {art['title'][:45]}: {exc}")
    time.sleep(REQUEST_DELAY)

t2_records = load_jsonl(TASK2_PATH)
print(f"\nTask 2 complete — {len(t2_records)} summaries saved to {TASK2_PATH}")
if errors_t2:
    print(f"  Errors: {len(errors_t2)} (see `errors_t2` list)")
if any(skipped_for_quota.values()):
    print(f"  Skipped for quota: {dict(skipped_for_quota)} — not marked done, will resume next run")

## Export to CSV
Produces two flat CSVs that are easier to work with in downstream analysis notebooks/scripts.

In [ ]:
df_t1 = pd.DataFrame(load_jsonl(TASK1_PATH))
df_t2 = pd.DataFrame(load_jsonl(TASK2_PATH))

csv_t1 = os.path.join(SUMMARIES_DIR, "task1_unconstrained.csv")
csv_t2 = os.path.join(SUMMARIES_DIR, "task2_constrained.csv")

df_t1.to_csv(csv_t1, index=False, encoding="utf-8")
df_t2.to_csv(csv_t2, index=False, encoding="utf-8")

print(f"Task 1 CSV : {csv_t1}  ({len(df_t1)} rows)")
print(f"Task 2 CSV : {csv_t2}  ({len(df_t2)} rows)")
print()
if len(df_t1):
    print("Task 1 breakdown (model \u00d7 article_dialect):")
    print(df_t1.groupby(["model", "article_dialect"])["summary"].count().unstack(fill_value=0))
if len(df_t2):
    print("\nTask 2 breakdown (model \u00d7 target_dialect):")
    print(df_t2.groupby(["model", "target_dialect"])["summary"].count().unstack(fill_value=0))

Task 1 CSV : ./summaries/task1_unconstrained.csv  (2 rows)
Task 2 CSV : ./summaries/task2_constrained.csv  (6 rows)

Task 1 breakdown (model × article_dialect):
article_dialect  AUS
model               
gemini             1
gpt_oss            1

Task 2 breakdown (model × target_dialect):
target_dialect  AUS  UK  US
model                      
gemini            1   1   1
gpt_oss           1   1   1


## Quick LSR Sanity Check
Applies the Lexical Switch Rate from `spelling_markers.py` to each summary as an immediate,
free (non-LLM) validity check — full analysis still belongs in `evaluate_on_summaries.py`.

- Task 1: a high LSR on a UK/AUS-sourced article hints at American-English drift.
- Task 2: a constrained-US summary should have LSR \u2248 1.0; constrained-UK/AUS should be \u2248 0.0.

In [ ]:
markers = load_spelling_markers()
print(f"Markers \u2014 US: {len(markers['US']):,}  UK: {len(markers['UK']):,}  AU: {len(markers['AU']):,}")


def compute_summary_lsr(summary: str) -> dict:
    return compute_lsr(strip_quotes(summary), markers)


df_t1 = pd.DataFrame(load_jsonl(TASK1_PATH))
if len(df_t1):
    lsr1 = df_t1["summary"].apply(compute_summary_lsr)
    df_t1["lsr"] = lsr1.apply(lambda r: r["lsr"])
    df_t1["total_dialect_tokens"] = lsr1.apply(lambda r: r["total_dialect_tokens"])
    df_t1.to_csv(csv_t1, index=False, encoding="utf-8")

    pivot1 = (
        df_t1[df_t1["total_dialect_tokens"] > 0]
        .groupby(["model", "article_dialect"])["lsr"]
        .agg(["mean", "count"]).round(4)
    )
    print("\nTask 1 \u2014 mean LSR by model \u00d7 source dialect (higher = more American English):")
    print(pivot1.to_string())
else:
    print("No Task 1 summaries yet.")

df_t2 = pd.DataFrame(load_jsonl(TASK2_PATH))
if len(df_t2):
    lsr2 = df_t2["summary"].apply(compute_summary_lsr)
    df_t2["lsr"] = lsr2.apply(lambda r: r["lsr"])
    df_t2["total_dialect_tokens"] = lsr2.apply(lambda r: r["total_dialect_tokens"])
    df_t2.to_csv(csv_t2, index=False, encoding="utf-8")

    pivot2 = (
        df_t2[df_t2["total_dialect_tokens"] > 0]
        .groupby(["model", "target_dialect"])["lsr"]
        .mean().unstack(fill_value=float("nan")).round(4)
    )
    print("\nTask 2 \u2014 mean LSR by model \u00d7 target dialect (US col \u2248 1.0, UK/AUS \u2248 0.0):")
    print(pivot2.to_string())
else:
    print("No Task 2 summaries yet.")

Markers — US: 2,766  UK: 1,730  AU: 2,786

Task 1 — mean LSR by model × source dialect (higher = more American English):
                         mean  count
model   article_dialect             
gemini  AUS               0.0      1
gpt_oss AUS               0.5      1

Task 2 — mean LSR by model × target dialect (US col ≈ 1.0, UK/AUS ≈ 0.0):
target_dialect  AUS   UK   US
model                        
gpt_oss         0.0  0.0  1.0


## Sanity Check — Sample Output
Prints one article with its unconstrained summaries from both models side by side.

In [ ]:
df_t1_check = pd.DataFrame(load_jsonl(TASK1_PATH))

if len(df_t1_check) == 0:
    print("No summaries yet \u2014 run Task 1 first.")
else:
    sample_title = df_t1_check.iloc[0]["article_title"]
    sample_rows  = df_t1_check[df_t1_check["article_title"] == sample_title]

    print(f"Article : {sample_title}")
    print(f"Topic   : {sample_rows.iloc[0]['topic']}")
    print(f"Outlet  : {sample_rows.iloc[0]['outlet']}  (dialect: {sample_rows.iloc[0]['article_dialect']})")
    print()
    for _, row in sample_rows.iterrows():
        print(f"\u2500\u2500\u2500 {row['model'].upper()} \u2500\u2500\u2500")
        print(textwrap.fill(row["summary"], width=90, subsequent_indent="  "))
        print()

Article : Hikers lost near Kosciuszko found by artificial intelligence drone
Topic   : artificial intelligence
Outlet  : abc_au  (dialect: AUS)

─── GEMINI ───
Two hikers lost in Kosciuszko National Park were rescued by an AI-powered drone, marking
  the first successful use of its AI detection system by Fire and Rescue NSW (FRNSW) for a
  missing hiker. The drone used thermal imaging and AI detection software to locate the
  men approximately 500 metres off track. Rescuers then used the drone's built-in speaker
  to contact the hikers and guide them out, less than five hours after they were reported
  missing. FRNSW stated the technology significantly reduced search time and risk for
  rescuers, with plans to improve it for various emergencies.

─── GPT_OSS ───
Two hikers in their early 20s who went missing on the Dead Horse Gap walking track in
  Kosciuszko National Park were located and rescued by a remote‑controlled drone equipped
  with artificial‑intelligence detection software. 

## Progress Summary
Made / needed / remaining for each task, plus a per-model breakdown — useful given
generation is spread across multiple runs due to daily API quotas.

In [ ]:
n_articles = len(all_articles)
n_models   = len(MODELS)
n_dialects = len(DIALECT_INSTRUCTIONS)

t1_records_now = load_jsonl(TASK1_PATH)
t2_records_now = load_jsonl(TASK2_PATH)

t1_made     = len(t1_records_now)
t1_needed   = n_articles * n_models
t1_remaining = t1_needed - t1_made

t2_made      = len(t2_records_now)
t2_needed    = n_articles * n_models * n_dialects
t2_remaining = t2_needed - t2_made

print("=" * 60)
print("PROGRESS SUMMARY")
print("=" * 60)
print(f"Task 1 (unconstrained) — made: {t1_made}  needed: {t1_needed}  remaining: {t1_remaining}")
print(f"Task 2 (constrained)   — made: {t2_made}  needed: {t2_needed}  remaining: {t2_remaining}")

t1_by_model = Counter(r["model"] for r in t1_records_now)
t2_by_model = Counter(r["model"] for r in t2_records_now)

print("\nBy model:")
for model in MODELS:
    m1 = t1_by_model.get(model, 0)
    m2 = t2_by_model.get(model, 0)
    print(f"  {model:<8} Task1: {m1}/{n_articles}   Task2: {m2}/{n_articles * n_dialects}")